In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed_train.csv')
print(df.shape)
df.head()

(1308, 220)


,MSSubClass,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,...,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,60,65,8450,4,4,3,7,5,2003,2003,...,0,0,0,0,1,0,0,0,1,0
1,20,80,9600,4,4,3,6,8,1976,1976,...,0,0,0,0,1,0,0,0,1,0
2,60,68,11250,3,4,3,7,5,2001,2002,...,0,0,0,0,1,0,0,0,1,0
3,70,60,9550,3,4,3,7,5,1915,1970,...,0,0,0,0,1,0,0,0,0,0
4,60,84,14260,3,4,3,8,5,2000,2000,...,0,0,0,0,1,0,0,0,1,0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2

X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (1046, 219), Test: (262, 219)


In [3]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
df[X.columns].corrwith(np.log1p(df['SalePrice'])).sort_values(ascending=False).head(10)

LivArea_Qual    0.843877
Bsmt_Qual       0.841526
Bath_Qual       0.812902
OverallQual     0.810769
Garage_Qual     0.792566
TotalSF_Qual    0.788954
GrLivArea       0.711354
GarageCars      0.678163
ExterQual       0.668031
KitchenQual     0.663898
dtype: float64

In [6]:
from sklearn.linear_model import LinearRegression

# use best single feature from preprocessing correlation analysis
X_train_simple = X_train[['LivArea_Qual']]
X_test_simple = X_test[['LivArea_Qual']]

slr = LinearRegression()
slr.fit(X_train_simple, y_train_log)

y_pred_slr = slr.predict(X_test_simple)

rmse = np.sqrt(MSE(y_test_log, y_pred_slr))
mae = MAE(y_test_log, y_pred_slr)
r2 = R2(y_test_log, y_pred_slr)

print(f'Simple Linear Regression (TotalSF_Qual)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Simple Linear Regression (TotalSF_Qual)
RMSE : 0.1985
MAE  : 0.1448
R²   : 0.7170


### Model 1: Simple Linear Regression

A single-feature linear regression model was trained using `LivArea_Qual` (above-ground
living area × overall quality) as the sole predictor, selected based on its highest
correlation (0.844) with the log-transformed target variable.

| Metric | Value |
|---|---|
| RMSE | 0.1985 |
| MAE | 0.1448 |
| R² | 0.7170 |

**Interpretation:**

- An R² of 0.717 means a single feature explains ~72% of the variance in sale price,
  which is strong for a one-variable model and reflects the high predictive power of
  the `LivArea_Qual` interaction term
- RMSE of 0.199 in log scale corresponds to roughly ±20% error in actual price terms
- This serves as the baseline — all subsequent models are evaluated against these metrics

**Limitations:**

- A single feature cannot capture the full complexity of housing prices
- The model ignores 219 other features including neighborhood, age, condition, and
  structural characteristics
- Expected to underfit — improvement is anticipated as more features are introduced

In [7]:
mlr = LinearRegression()
mlr.fit(X_train, y_train_log)

y_pred_mlr = mlr.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_mlr))
mae = MAE(y_test_log, y_pred_mlr)
r2 = R2(y_test_log, y_pred_mlr)

print(f'Multiple Linear Regression')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Multiple Linear Regression
RMSE : 0.1335
MAE  : 0.0864
R²   : 0.8720


### Model 2: Multiple Linear Regression

Multiple Linear Regression was trained using all 220 features in the dataset,
extending the single-feature baseline to capture the full breadth of available
property information.

| Metric | SLR (Baseline) | MLR | Change |
|---|---|---|---|
| RMSE | 0.1985 | 0.1340 | ↓ 32% |
| MAE | 0.1448 | 0.0876 | ↓ 39% |
| R² | 0.7170 | 0.8710 | ↑ 15% |

**Interpretation:**

- R² of 0.871 indicates the model explains ~87% of variance in sale price —
  a significant improvement over the single-feature baseline
- RMSE of 0.134 in log scale corresponds to roughly ±13% error in actual price terms
- The large improvement confirms that the additional 219 features carry genuine
  predictive signal beyond living area and quality alone

**Limitations:**

- With 220 features and only 1046 training samples, the model is at risk of
  overfitting — it may be fitting noise in the training data rather than
  generalizing real patterns
- Multicollinearity among engineered interaction features can inflate and
  destabilize individual coefficients, making interpretation unreliable
- These issues motivate the use of regularized linear models in the next step

In [8]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

alphas = [0.01, 0.1, 1, 10, 50, 100, 200, 500, 1000]

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    scores = cross_val_score(ridge, X_train, y_train_log, cv=5, scoring='r2')
    print(f'alpha={alpha:6} | R²: {scores.mean():.4f} ± {scores.std():.4f}')

alpha=  0.01 | R²: 0.8865 ± 0.0285
alpha=   0.1 | R²: 0.8903 ± 0.0245
alpha=     1 | R²: 0.9020 ± 0.0142
alpha=    10 | R²: 0.9110 ± 0.0136
alpha=    50 | R²: 0.9069 ± 0.0159
alpha=   100 | R²: 0.9017 ± 0.0174
alpha=   200 | R²: 0.8953 ± 0.0191
alpha=   500 | R²: 0.8862 ± 0.0215
alpha=  1000 | R²: 0.8789 ± 0.0234


In [9]:
ridge = Ridge(alpha=10)
ridge.fit(X_train, y_train_log)

y_pred_ridge = ridge.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_ridge))
mae = MAE(y_test_log, y_pred_ridge)
r2 = R2(y_test_log, y_pred_ridge)

print(f'Ridge Regression (alpha=10)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Ridge Regression (alpha=10)
RMSE : 0.1311
MAE  : 0.0846
R²   : 0.8765


### Model 3: Ridge Regression

Ridge Regression was applied to address the overfitting risk and multicollinearity
identified in the Multiple Linear Regression model. Ridge adds an L2 penalty term
to the loss function, shrinking large coefficients toward zero without eliminating
any features entirely.

**Hyperparameter Tuning:**

Cross-validation was performed across a range of alpha values (regularization strength)
to identify the optimal penalty. A 5-fold cross-validation was used to evaluate each
candidate.

| Alpha | CV R² | Std Dev |
|---|---|---|
| 0.01 | 0.8854 | ±0.0310 |
| 0.1 | 0.8896 | ±0.0266 |
| 1 | 0.9016 | ±0.0147 |
| 10 | 0.9111 | ±0.0132 |
| 50 | 0.9072 | ±0.0157 |
| 100 | 0.9020 | ±0.0172 |
| 200 | 0.8956 | ±0.0188 |
| 500 | 0.8863 | ±0.0212 |
| 1000 | 0.8789 | ±0.0232 |

`alpha=10` was selected as the optimal value — highest mean R² with the lowest
standard deviation, indicating both strong performance and stable generalization.

**Results:**

| Metric | SLR | MLR | Ridge |
|---|---|---|---|
| RMSE | 0.1985 | 0.1340 | 0.1317 |
| MAE | 0.1448 | 0.0876 | 0.0854 |
| R² | 0.7170 | 0.8710 | 0.8754 |

**Interpretation:**

- Ridge marginally outperforms MLR on RMSE and MAE while maintaining comparable R²
- The cross-validation R² of 0.911 at alpha=10 is notably higher than the test R²
  of MLR, confirming that Ridge generalizes better despite similar test metrics
- The L2 penalty successfully stabilized inflated coefficients caused by
  multicollinearity among engineered interaction features
- The marginal test R² difference between MLR and Ridge suggests MLR was overfitting
  slightly — Ridge corrected this by shrinking unstable coefficients

**Key Takeaway:**

Ridge does not dramatically outperform MLR on this test set, but it is the more
reliable model — its cross-validation performance is stronger and its coefficients
are more stable and interpretable.

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
from sklearn.linear_model import Lasso

alphas = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1]

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    scores = cross_val_score(lasso, X_train_scaled, y_train_log, cv=5, scoring='r2')
    print(f'alpha={alpha:.4f} | R²: {scores.mean():.4f} ± {scores.std():.4f}')

alpha=0.0001 | R²: 0.8907 ± 0.0257
alpha=0.0005 | R²: 0.9055 ± 0.0131
alpha=0.0010 | R²: 0.9073 ± 0.0146
alpha=0.0050 | R²: 0.9122 ± 0.0144
alpha=0.0100 | R²: 0.9036 ± 0.0167
alpha=0.0500 | R²: 0.8199 ± 0.0373
alpha=0.1000 | R²: 0.7165 ± 0.0439


In [12]:
lasso = Lasso(alpha=0.005, max_iter=10000)
lasso.fit(X_train_scaled, y_train_log)
y_pred_lasso = lasso.predict(X_test_scaled)

rmse = np.sqrt(MSE(y_test_log, y_pred_lasso))
mae = MAE(y_test_log, y_pred_lasso)
r2 = R2(y_test_log, y_pred_lasso)

zeroed = (lasso.coef_ == 0).sum()

print(f'Lasso Regression (alpha=0.005)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')
print(f'Features zeroed out: {zeroed} / {X_train.shape[1]}')

Lasso Regression (alpha=0.005)
RMSE : 0.1315
MAE  : 0.0829
R²   : 0.8758
Features zeroed out: 158 / 219


### Model 4: Lasso Regression

Lasso Regression extends Ridge by applying an L1 penalty instead of L2. Unlike Ridge,
which shrinks all coefficients toward zero, Lasso can drive coefficients to exactly zero —
effectively performing automatic feature selection alongside regularization.

**Note on Feature Scaling:**

Unlike SLR, MLR, and Ridge which were trained on unscaled features, Lasso requires
scaled features. Lasso solves using an iterative algorithm called coordinate descent,
which is sensitive to feature scale — unscaled features caused convergence warnings
at lower alpha values, indicating the optimizer was failing to find a stable solution
within the iteration limit. Applying StandardScaler resolved this by bringing all
features to the same scale, ensuring the penalty is applied fairly and the optimizer
converges cleanly.

**Hyperparameter Tuning:**

Cross-validation was performed across a range of alpha values using scaled features.

| Alpha | CV R² | Std Dev |
|---|---|---|
| 0.0001 | 0.8903 | ±0.0273 |
| 0.0005 | 0.9058 | ±0.0129 |
| 0.0010 | 0.9077 | ±0.0144 |
| 0.0050 | 0.9123 | ±0.0144 |
| 0.0100 | 0.9036 | ±0.0167 |
| 0.0500 | 0.8199 | ±0.0373 |
| 0.1000 | 0.7165 | ±0.0439 |

`alpha=0.005` was selected — highest CV R² (0.9123) with low and stable standard
deviation. The sharp drop beyond alpha=0.01 indicates over-regularization, where
too many features are zeroed out and useful signal is lost.

**Results:**

| Metric | SLR | MLR | Ridge | Lasso |
|---|---|---|---|---|
| RMSE | 0.1985 | 0.1340 | 0.1317 | 0.1316 |
| MAE | 0.1448 | 0.0876 | 0.0854 | 0.0830 |
| R² | 0.7170 | 0.8710 | 0.8754 | 0.8756 |

**Key Finding — Automatic Feature Selection:**

Lasso zeroed out 158 out of 220 features, retaining only 62 as active predictors.
This confirms that the majority of one-hot encoded nominal features contribute
negligible signal, and the dataset's predictive power is concentrated in a small
subset of structural, quality, and area-based features.

**Interpretation:**

- Lasso marginally outperforms Ridge on all three metrics despite using only 62 features
- The sparser model generalizes slightly better, suggesting the 158 zeroed features
  were adding noise rather than signal in the linear setting
- Linear models appear to have reached their ceiling at ~0.875 R² on this dataset,
  as the underlying relationships between features and price are inherently non-linear
- This motivates the transition to tree-based models, which can capture conditional
  and non-linear patterns without assumptions about feature relationships

**Key Takeaway:**

Lasso is the strongest linear model in this pipeline — it matches Ridge's performance
while producing a far more interpretable and compact model. The feature zeroing
provides direct insight into which variables matter in a linear context.

In [13]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train[['LivArea_Qual']])
X_test_poly = poly.transform(X_test[['LivArea_Qual']])

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train_log)
y_pred_poly = poly_model.predict(X_test_poly)

rmse = np.sqrt(MSE(y_test_log, y_pred_poly))
mae = MAE(y_test_log, y_pred_poly)
r2 = R2(y_test_log, y_pred_poly)

print(f'Polynomial Regression (degree=2, LivArea_Qual)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Polynomial Regression (degree=2, LivArea_Qual)
RMSE : 0.1990
MAE  : 0.1452
R²   : 0.7155


In [14]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

max_depths = [3, 5, 7, 10, 15, 20, None]

for depth in max_depths:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    scores = cross_val_score(dt, X_train, y_train_log, cv=5, scoring='r2')
    print(f'max_depth={str(depth):5} | R²: {scores.mean():.4f} ± {scores.std():.4f}')

max_depth=3     | R²: 0.6896 ± 0.0703
max_depth=5     | R²: 0.7265 ± 0.0626
max_depth=7     | R²: 0.7201 ± 0.0736
max_depth=10    | R²: 0.7045 ± 0.0693
max_depth=15    | R²: 0.6997 ± 0.0692
max_depth=20    | R²: 0.7009 ± 0.0687
max_depth=None  | R²: 0.7062 ± 0.0680


In [15]:
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train_log)
y_pred_dt = dt.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_dt))
mae = MAE(y_test_log, y_pred_dt)
r2 = R2(y_test_log, y_pred_dt)

print(f'Decision Tree (max_depth=5)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Decision Tree (max_depth=5)
RMSE : 0.1826
MAE  : 0.1324
R²   : 0.7604


### Model 5: Decision Tree Regressor

A Decision Tree Regressor was trained as the first tree-based model in the pipeline.
Decision trees learn by recursively splitting the data based on feature thresholds,
creating a set of if-else rules that map feature values to predicted outcomes.

**Hyperparameter Tuning:**

`max_depth` controls how many splits the tree can make. Too shallow and it underfits,
too deep and it memorizes the training data.

| max_depth | CV R² | Std Dev |
|---|---|---|
| 3 | 0.6893 | ±0.0703 |
| 5 | 0.7317 | ±0.0696 |
| 7 | 0.7218 | ±0.0765 |
| 10 | 0.7132 | ±0.0781 |
| 15 | 0.7065 | ±0.0849 |
| 20 | 0.7028 | ±0.0806 |
| None | 0.6999 | ±0.0794 |

`max_depth=5` was selected — best CV R² with relatively stable variance.

**Results:**

| Metric | SLR | MLR | Ridge | Lasso | Decision Tree |
|---|---|---|---|---|---|
| RMSE | 0.1985 | 0.1340 | 0.1317 | 0.1316 | 0.1826 |
| MAE | 0.1448 | 0.0876 | 0.0854 | 0.0830 | 0.1324 |
| R² | 0.7170 | 0.8710 | 0.8754 | 0.8756 | 0.7604 |

**Interpretation:**

- Decision Tree performs worse than all linear models despite being a more complex
  model — R² of 0.760 vs Lasso's 0.876
- With `max_depth=5`, the tree makes only 5 splits across 220 features, effectively
  using a tiny fraction of available information
- Increasing depth does not help — CV R² drops consistently beyond depth 5,
  indicating the tree overfits to training noise rather than learning generalizable patterns
- High variance across CV folds (±0.07) compared to linear models (±0.013) confirms
  that single decision trees are unstable and sensitive to the specific training samples

**Why Decision Trees Underperform Here:**

A single tree is a high-variance, low-bias model in isolation. It makes greedy,
locally optimal splits without considering the global structure of the data. On a
dataset with 220 features and complex interactions, a single tree cannot capture
the full signal without overfitting.

**Key Takeaway:**

The Decision Tree result motivates the use of ensemble methods. Random Forest and
XGBoost were specifically designed to overcome the instability of single trees by
combining many trees — averaging out variance and capturing complex patterns that
no single tree can learn reliably.

In [16]:
from sklearn.ensemble import RandomForestRegressor

n_estimators = [50, 100, 200]
max_depths = [10, 20, None]

for n in n_estimators:
    for depth in max_depths:
        rf = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=42, n_jobs=-1)
        scores = cross_val_score(rf, X_train, y_train_log, cv=5, scoring='r2')
        print(f'n={n:3}, depth={str(depth):5} | R²: {scores.mean():.4f} ± {scores.std():.4f}')

n= 50, depth=10    | R²: 0.8585 ± 0.0262
n= 50, depth=20    | R²: 0.8599 ± 0.0259
n= 50, depth=None  | R²: 0.8596 ± 0.0262
n=100, depth=10    | R²: 0.8602 ± 0.0236
n=100, depth=20    | R²: 0.8615 ± 0.0240
n=100, depth=None  | R²: 0.8612 ± 0.0241
n=200, depth=10    | R²: 0.8602 ± 0.0244
n=200, depth=20    | R²: 0.8620 ± 0.0240
n=200, depth=None  | R²: 0.8619 ± 0.0240


In [17]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 0.5]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train_log)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV R²: {grid_search.best_score_:.4f}')

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best CV R²: 0.8770


In [18]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train_log)
y_pred_rf = rf.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_rf))
mae = MAE(y_test_log, y_pred_rf)
r2 = R2(y_test_log, y_pred_rf)

print(f'Random Forest (tuned)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Random Forest (tuned)
RMSE : 0.1490
MAE  : 0.0950
R²   : 0.8404


In [19]:
import xgboost as xgb

params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
}

xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)
grid_search_xgb = GridSearchCV(xgb_model, params, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_search_xgb.fit(X_train, y_train_log)

print(f'Best params: {grid_search_xgb.best_params_}')
print(f'Best CV R²: {grid_search_xgb.best_score_:.4f}')

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
Best CV R²: 0.8996


In [20]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train_log)
y_pred_xgb = xgb_model.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_xgb))
mae = MAE(y_test_log, y_pred_xgb)
r2 = R2(y_test_log, y_pred_xgb)

print(f'XGBoost (tuned)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

XGBoost (tuned)
RMSE : 0.1388
MAE  : 0.0911
R²   : 0.8616
